In [1]:
import sys
from pathlib import Path

# Add project root and experiments to path
PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

# Import paths
from paths import (
    UMU_SYNTH_DIR,
    UMU_SYNTH_TOMOS_DIR,
    UMU_SYNTH_CSV,
    TOMOTWIN_MODEL_FILE,
    PROPICKER_TOOLS_DIR,
    PROPICKER_MODEL_FILE,
    EXP3_RESULTS_DIR,
    EXP3_CHECKPOINTS_DIR,
    EXP3_COORDS_DIR,
    EXP4_RESULTS_DIR,
    EXP4_DATA_DIR,
    EXP4_PROMPTS_DIR,
    EXP4_INFERENCE_DIR,
    EXP4_COORDS_DIR,
)

# Import config
from experiments.config import (
    setup_propicker_paths,
    THYROGLOBULIN_NAME,
    THYROGLOBULIN_LABEL,
    PROMPT_SIZE,
    PROMPT_HALF,
    THYROGLOBULIN_DIAMETER,
    EXP4_VAL_TOMOS,
    EXP4_TRAIN_POOL,
    EXP4_NUM_PROMPTS,
    EXP4_MIN_ANGULAR_DISTANCE,
    EXP3_INCREMENTS,
)

# Setup ProPicker imports
PROPICKER_DIR = setup_propicker_paths()

import subprocess
import json
import pandas as pd
import numpy as np
import torch
from matplotlib import pyplot as plt
import seaborn as sns
from scipy.spatial.transform import Rotation
from scipy.spatial.distance import cdist

from inference.tomotwin import pass_subtomos_through_tomotwin
from utils.mrctools import load_mrc_data

import warnings
warnings.filterwarnings("ignore")

# Create output directories
EXP4_DATA_DIR.mkdir(parents=True, exist_ok=True)
EXP4_PROMPTS_DIR.mkdir(parents=True, exist_ok=True)
EXP4_COORDS_DIR.mkdir(parents=True, exist_ok=True)

print(f"ProPicker tools: {PROPICKER_DIR}")
print(f"UMU Synth data: {UMU_SYNTH_DIR}")
print(f"EXP3 checkpoints: {EXP3_CHECKPOINTS_DIR}")
print(f"EXP4 results: {EXP4_RESULTS_DIR}")
print(f"EXP4 data dir: {EXP4_DATA_DIR}")
print(f"\nExperiment parameters:")
print(f"  Validation tomograms: {EXP4_VAL_TOMOS}")
print(f"  Number of prompts: {EXP4_NUM_PROMPTS}")
print(f"  Min angular distance: {EXP4_MIN_ANGULAR_DISTANCE}°")
print(f"  Prompt size: {PROMPT_SIZE}×{PROMPT_SIZE}×{PROMPT_SIZE}")

ProPicker tools: /home/carloshg/Dev/cryoet-particle-picking/tools/ProPicker/propicker
UMU Synth data: /home/carloshg/Dev/cryoet-particle-picking/data/umu_synth
EXP3 checkpoints: /home/carloshg/Dev/cryoet-particle-picking/results/exp3_ppicker_limits/checkpoints
EXP4 results: /home/carloshg/Dev/cryoet-particle-picking/results/exp4_ppicker_rotations
EXP4 data dir: /home/carloshg/Dev/cryoet-particle-picking/results/exp4_ppicker_rotations/data

Experiment parameters:
  Validation tomograms: ['tomo_rec_5_snr1.66', 'tomo_rec_6_snr1.17', 'tomo_rec_7_snr1.13', 'tomo_rec_8_snr0.57', 'tomo_rec_9_snr1.28']
  Number of prompts: 10
  Min angular distance: 30.0°
  Prompt size: 37×37×37


In [2]:
import mrcfile
import numpy as np
# Load inference results for multi_inc16 prompts
# Each prompt has segmentation outputs for validation tomograms

NUM_PROMPTS = 10
INFERENCE_TYPE = "multi_inc16"

# Dictionary to store all results: {prompt_idx: {tomo_name: tensor}}
multi16_results = {}

for prompt_idx in range(NUM_PROMPTS):
    prompt_dir = EXP4_INFERENCE_DIR / f"{INFERENCE_TYPE}_prompt{prompt_idx}" / "full_segmentation_output"
    
    if not prompt_dir.exists():
        print(f"Warning: Directory not found: {prompt_dir}")
        continue
    
    multi16_results[prompt_idx] = {}
    
    # Load all .pt files in this prompt's inference directory
    pt_files = sorted(prompt_dir.glob("*.pt"))
    
    for pt_file in pt_files:
        tomo_name = pt_file.stem  # e.g., "tomo_rec_5_snr1.66"
        tensor_data = torch.load(pt_file, map_location='cpu')
        multi16_results[prompt_idx][tomo_name] = tensor_data
        new_name = f'{prompt_dir}/{tomo_name}.mrc'
        
        # Convert tensor to numpy (PyTorch uses .numpy(), not .to_numpy())
        tensor_np = tensor_data.numpy().astype(np.float32)
        
        with mrcfile.new(new_name, overwrite=True) as mrc:
            mrc.set_data(tensor_np)
            mrc.voxel_size = (10.0124, 10.0124, 10.0124)
            
        
    print(f"Prompt {prompt_idx}: Loaded and saved {len(pt_files)} tomogram segmentations")

# Summary
print(f"\n--- Summary ---")
print(f"Total prompts loaded: {len(multi16_results)}")
if multi16_results:
    first_prompt = list(multi16_results.keys())[0]
    print(f"Tomograms per prompt: {list(multi16_results[first_prompt].keys())}")
    
    # Show shape of first tensor
    first_tomo = list(multi16_results[first_prompt].keys())[0]
    tensor_shape = multi16_results[first_prompt][first_tomo].shape
    print(f"Tensor shape (first example): {tensor_shape}")

Prompt 0: Loaded and saved 5 tomogram segmentations
Prompt 1: Loaded and saved 5 tomogram segmentations
Prompt 2: Loaded and saved 5 tomogram segmentations
Prompt 3: Loaded and saved 5 tomogram segmentations
Prompt 4: Loaded and saved 5 tomogram segmentations
Prompt 5: Loaded and saved 5 tomogram segmentations
Prompt 6: Loaded and saved 5 tomogram segmentations
Prompt 7: Loaded and saved 5 tomogram segmentations
Prompt 8: Loaded and saved 5 tomogram segmentations
Prompt 9: Loaded and saved 5 tomogram segmentations

--- Summary ---
Total prompts loaded: 10
Tomograms per prompt: ['tomo_rec_5_snr1.66', 'tomo_rec_6_snr1.17', 'tomo_rec_7_snr1.13', 'tomo_rec_8_snr0.57', 'tomo_rec_9_snr1.28']
Tensor shape (first example): torch.Size([196, 642, 642])
